# Imputation Methods — Interactive Demonstration

This notebook visually compares all 13 imputation methods on a single Kepler light curve
at three missingness levels (10%, 30%, 50%). It reproduces the lightcurve demonstration
figure type from the thesis introduction.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.gap_injection import inject_gaps
from src.imputation.registry import get_all_imputers, ALL_METHOD_NAMES
from src.evaluation.metrics import rmse as compute_rmse, mae as compute_mae

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

DATA_DIR = Path('../data/processed')
files = sorted(DATA_DIR.glob('*.parquet'))
print(f'Found {len(files)} light curves')

In [ ]:
# Load one example light curve
if not files:
    # Generate synthetic example for demonstration
    rng = np.random.default_rng(42)
    N = 600
    t = np.linspace(0, 25, N)
    flux = 1.0 + 0.25 * np.sin(2*np.pi*t/3.5) + 0.08 * np.sin(2*np.pi*t/1.2) + 0.03*rng.standard_normal(N)
    label = 0
    print('Using synthetic light curve (no data directory found)')
else:
    df = pd.read_parquet(files[0])
    t = df['time'].values[:600]
    flux = df['flux'].values[:600]
    label = int(df['class_label'].iloc[0])
    print(f'Loaded: {files[0].name}, class={label}, N={len(t)}')

In [ ]:
# Set up imputers (use small subset for speed)
DEMO_METHODS = ['Mean_Fill', 'Forward_Fill', 'Linear', 'Spline', 'GP_Matern32', 'TS_MICE',
                'KNN_Impute', 'RF_Impute', 'MF_Impute', 'GB_MICE', 'SAITS']
imputers = get_all_imputers(seed=42)

FRAC = 0.30
SEED = 7

gapped, mask, ground_truth = inject_gaps(flux, p=FRAC, seed=SEED)
missing_idx = np.where(~mask)[0]

results = {}
for name in DEMO_METHODS:
    if name not in imputers:
        continue
    imp = imputers[name]
    print(f'Running {name}...', end=' ', flush=True)
    try:
        flux_imp = imp.impute(gapped.copy(), mask, t)
        rmse_val = compute_rmse(ground_truth, flux_imp[missing_idx])
        mae_val  = compute_mae(ground_truth, flux_imp[missing_idx])
        results[name] = {'flux': flux_imp, 'rmse': rmse_val, 'mae': mae_val}
        print(f'RMSE={rmse_val:.4f}')
    except Exception as e:
        print(f'FAILED: {e}')

In [ ]:
# Plot all methods on a short window
WIN = slice(0, 200)
n_methods = len(results)
ncols = 3
nrows = (n_methods + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows), sharex=True)
axes = axes.flat

t_w = t[WIN] - t[WIN][0]
f_w = flux[WIN]
g_w = gapped[WIN]
m_w = mask[WIN]

for i, (name, res) in enumerate(results.items()):
    ax = axes[i]
    f_imp_w = res['flux'][WIN]

    ax.plot(t_w, f_w, 'k-', lw=0.7, alpha=0.4, label='True')
    ax.plot(t_w[m_w], g_w[m_w], 'k.', ms=2, label='Observed')
    ax.plot(t_w[~m_w], f_imp_w[~m_w], 'r.', ms=3, alpha=0.8, label='Imputed')

    ax.set_title(f'{name}\nRMSE={res["rmse"]:.4f}  MAE={res["mae"]:.4f}', fontsize=8)
    ax.set_ylabel('Flux')
    if i == 0:
        ax.legend(fontsize=7)

# Hide unused axes
for j in range(i+1, nrows*ncols):
    axes[j].set_visible(False)

fig.suptitle(f'All imputation methods — 30% missingness (window: 0–200 cadences)', fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# RMSE comparison bar chart
names = list(results.keys())
rmse_vals = [results[n]['rmse'] for n in names]
mae_vals  = [results[n]['mae']  for n in names]

x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(x, rmse_vals, color='steelblue', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('RMSE'); axes[0].set_title('RMSE by method (30% missingness)')

axes[1].bar(x, mae_vals, color='darkorange', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('MAE'); axes[1].set_title('MAE by method (30% missingness)')

fig.tight_layout()
plt.show()